# 33 — Train / Validation / Test

### Why this exists

Evaluation is only meaningful when the split matches how future data will arrive.

### Learning objectives

By the end you should be able to:
- explain the mechanism in plain language;
- predict a system behavior before running the experiment;
- measure one observable consequence;
- diagnose one controlled failure;
- state the production implication.

### Prerequisite

Use the preceding notebook in this sequence as your conceptual starting point. You do not need a vendor account for this notebook.

### Engineering loop

**Why → Sketch → Predict → Build → Measure → Break → Diagnose → Improve → Production**


## Capability contract

**Capability:** C07 — ML Reliability  
**Workstream:** ML Platform  
**Primary roles:** ML Engineer|Data Engineer  
**You will prove:** Separate offline signal from production behavior and define validation


## 🖊️ Sketch note

![Sketch note — Train / Validation / Test](../assets/sketch-notes/33-train-validation-test.svg)

> **Visual-first rule:** explain the system in your own words before reading the code.

## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S12 — 1145: DATA-1145 — Model review cannot reproduce the reported test score** → [`../work_simulations/12_acme_commerce_evaluation_split/README.md`](../work_simulations/12_acme_commerce_evaluation_split/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


## Predict before you run

**Question:** Will a random split leak future information into the past?

Write your prediction and the reason in your own words before running the code.


In [ ]:
import pandas as pd
df=pd.DataFrame({"day":range(1,11),"label":[0,1,0,0,1,0,1,0,1,0]})
train=df[df.day<=6]; valid=df[(df.day>=7)&(df.day<=8)]; test=df[df.day>=9]
print(len(train),len(valid),len(test))
assert train.day.max()<valid.day.min()<test.day.min()


## Measure

Record the output that tests your prediction. If you can, change exactly one input and compare before/after behavior.


## Break deliberately

Introduce one controlled failure: malformed input, stale data, wrong split, missing context, unauthorized tool, or another failure relevant to this notebook. Keep the failure small and observable.


In [ ]:
# Break: create a random-style split that mixes future rows into training.
train_bad=df.sample(frac=0.6,random_state=42)
leak_possible=not (train_bad.day.max()<df.drop(train_bad.index).day.min())
print("future information can enter training:",leak_possible)
assert leak_possible


## Diagnose

Write:
1. **Symptom** — what changed?
2. **Evidence** — what observation supports it?
3. **Mechanism** — why did it happen?
4. **Fix** — what is the smallest defensible improvement?


## Improve

Make the smallest change that prevents or exposes the failure. Re-run the same measurement so the comparison is fair.


## Production bridge

A production system also needs ownership, versioning, observability, security/authorization, cost or latency budgets, failure recovery and a rollback path. Name the two that matter most here and why.


## Explain it in 60–90 seconds

Explain the mechanism, one surprising failure mode, and one production trade-off without relying on vendor names.


## Independent challenge

Build a second experiment around **time-aware evaluation**. Change one assumption, measure the result, and defend the design choice.

### Evidence to keep

- prediction
- measurement
- failure evidence
- diagnosis
- improvement
- production implication


## Compare split strategies

In [ ]:
import numpy as np
x=np.arange(12)
random_train=set(x[[0,2,4,6,8,10]])
time_train=set(x[:6])
print("random train max:",max(random_train),"time train max:",max(time_train))
assert max(time_train)<min(set(x)-time_train)

## Break temporal assumptions

In [ ]:
test_period=set(x[6:])
leak=max(time_train)>=min(test_period)
print("train/test time overlap:",leak)
assert not leak